# ✈️ Airline Review Intelligence Platform
### Powered by Snowflake Cortex | End-to-End NLP Pipeline

| Phase | Description |
|-------|-------------|
| **0** | Environment setup |
| **1** | Data ingestion — load CSV from stage |
| **2** | Harmonize — clean, parse dates, compute aspect averages |
| **3** | Translate — multilingual reviews → English |
| **4a** | Sentiment scoring — Cortex Sentiment (–1 to +1) |
| **4b** | Classify — recommend intent + text rating |
| **5a** | Aspect-based sentiment — JSON via Cortex Complete |
| **5b** | Issue identification — per-airline LLM summary |
| **6** | Analytics views + visualisations for Streamlit |

> **Prerequisites:** Snowflake account with Cortex LLM functions enabled in your region.  
> Upload `Airline_review.csv` to the internal stage before running Phase 1.  
> Supported regions: https://docs.snowflake.com/user-guide/snowflake-cortex/llm-functions#availability

## Setup — Imports & Snowpark Session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.cortex import Translate, Sentiment, Complete

session = get_active_session()
print("Session active:", session.get_current_database(), "/", session.get_current_schema())

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "DejaVu Sans", "font.size": 11,
})
PALETTE = ["#1D9E75","#E24B4A","#3B8BD4","#EF9F27","#7F77DD","#D85A30","#639922"]
print("Libraries loaded")

---
## Phase 0 — Environment Setup
Create database, schemas, and warehouse. Skip if already created.

In [ ]:
for stmt in [
    "CREATE DATABASE IF NOT EXISTS airline_reviews_db",
    "CREATE SCHEMA IF NOT EXISTS airline_reviews_db.raw",
    "CREATE SCHEMA IF NOT EXISTS airline_reviews_db.harmonized",
    "CREATE SCHEMA IF NOT EXISTS airline_reviews_db.analytics",
    "CREATE SCHEMA IF NOT EXISTS airline_reviews_db.cortex_output",
    """CREATE WAREHOUSE IF NOT EXISTS airline_ds_wh
        WAREHOUSE_SIZE = 'medium' AUTO_SUSPEND = 120
        AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE""",
    "USE DATABASE airline_reviews_db",
    "USE SCHEMA raw",
    "USE WAREHOUSE airline_ds_wh",
]:
    session.sql(stmt.strip()).collect()

print("Environment ready")
print("Database :", session.get_current_database())
print("Warehouse:", session.get_current_warehouse())

---
## Phase 1 — Data Ingestion

**Before running**, upload `Airline_review.csv` to the Snowflake stage.

**SnowSQL CLI:**
```bash
PUT file:///path/to/Airline_review.csv @airline_reviews_db.public.airline_stage;
```
**Snowsight UI:** `Data` → `Add Data` → `Load files into a Stage` → select `airline_stage`

In [ ]:
session.sql("""
    CREATE OR REPLACE FILE FORMAT airline_reviews_db.public.csv_ff
        TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        SKIP_HEADER = 1 NULL_IF = ('','NULL','null')
        EMPTY_FIELD_AS_NULL = TRUE TRIM_SPACE = TRUE
""").collect()

session.sql("""
    CREATE OR REPLACE STAGE airline_reviews_db.public.airline_stage
        FILE_FORMAT = airline_reviews_db.public.csv_ff
""").collect()

session.sql("""
    CREATE OR REPLACE TABLE airline_reviews_db.raw.airline_reviews_raw (
        row_id NUMBER, airline_name VARCHAR,
        overall_rating NUMBER(3,1), review_title VARCHAR,
        review_date VARCHAR, verified VARCHAR, review VARCHAR,
        aircraft VARCHAR, type_of_traveller VARCHAR, seat_type VARCHAR,
        route VARCHAR, date_flown VARCHAR,
        seat_comfort NUMBER(3,1), cabin_staff_service NUMBER(3,1),
        food_and_beverages NUMBER(3,1), ground_service NUMBER(3,1),
        inflight_entertainment NUMBER(3,1), wifi_and_connectivity NUMBER(3,1),
        value_for_money NUMBER(3,1), recommended VARCHAR
    )
""").collect()

session.sql("""
    COPY INTO airline_reviews_db.raw.airline_reviews_raw
    FROM @airline_reviews_db.public.airline_stage
    FILE_FORMAT = (FORMAT_NAME = 'airline_reviews_db.public.csv_ff')
    ON_ERROR = 'CONTINUE'
""").collect()

n = session.sql("SELECT COUNT(*) AS n FROM airline_reviews_db.raw.airline_reviews_raw").collect()[0]["N"]
print(f"Ingestion complete — {n:,} rows loaded")

---
## Phase 2 — Harmonized Layer
Parse dates, cast types, and compute `avg_aspect_score` from the 7 structured rating columns.

In [ ]:
session.sql("""
    CREATE OR REPLACE VIEW airline_reviews_db.harmonized.airline_reviews_v AS
    SELECT
        row_id AS review_id,
        TRIM(airline_name) AS airline_name,
        overall_rating,
        TRIM(review_title) AS review_title,
        TRY_TO_DATE(REGEXP_REPLACE(review_date,'(\\d+)(st|nd|rd|th)','\\1'),'DD MMMM YYYY') AS review_date,
        (verified = 'True') AS verified,
        TRIM(review) AS review,
        NULLIF(TRIM(aircraft),'') AS aircraft,
        TRIM(type_of_traveller) AS traveller_type,
        TRIM(seat_type) AS seat_type,
        TRIM(route) AS route,
        seat_comfort, cabin_staff_service, food_and_beverages,
        ground_service, inflight_entertainment, wifi_and_connectivity,
        value_for_money,
        LOWER(TRIM(recommended)) AS recommended,
        ROUND(
            (COALESCE(seat_comfort,0)+COALESCE(cabin_staff_service,0)+
             COALESCE(food_and_beverages,0)+COALESCE(ground_service,0)+
             COALESCE(inflight_entertainment,0)+COALESCE(wifi_and_connectivity,0)+
             COALESCE(value_for_money,0))
            / NULLIF(
                (CASE WHEN seat_comfort IS NOT NULL THEN 1 ELSE 0 END+
                 CASE WHEN cabin_staff_service IS NOT NULL THEN 1 ELSE 0 END+
                 CASE WHEN food_and_beverages IS NOT NULL THEN 1 ELSE 0 END+
                 CASE WHEN ground_service IS NOT NULL THEN 1 ELSE 0 END+
                 CASE WHEN inflight_entertainment IS NOT NULL THEN 1 ELSE 0 END+
                 CASE WHEN wifi_and_connectivity IS NOT NULL THEN 1 ELSE 0 END+
                 CASE WHEN value_for_money IS NOT NULL THEN 1 ELSE 0 END),0)
        ,2) AS avg_aspect_score
    FROM airline_reviews_db.raw.airline_reviews_raw
    WHERE review IS NOT NULL AND TRIM(review) != ''
""").collect()

session.sql("CREATE OR REPLACE VIEW airline_reviews_db.analytics.airline_reviews_v AS SELECT * FROM airline_reviews_db.harmonized.airline_reviews_v").collect()

df_prev = session.sql("SELECT airline_name, overall_rating, seat_type, traveller_type, avg_aspect_score, LEFT(review,80) AS snippet FROM airline_reviews_db.harmonized.airline_reviews_v LIMIT 5").to_pandas()
print("Harmonized view created")
df_prev

---
## Exploratory Data Analysis
Quick overview of the dataset before Cortex AI.

In [ ]:
df_dist = session.sql("SELECT overall_rating, COUNT(*) AS cnt FROM airline_reviews_db.harmonized.airline_reviews_v GROUP BY overall_rating ORDER BY overall_rating").to_pandas()
df_top  = session.sql("SELECT airline_name, COUNT(*) AS reviews, ROUND(AVG(overall_rating),2) AS avg_rating FROM airline_reviews_db.harmonized.airline_reviews_v GROUP BY airline_name ORDER BY reviews DESC LIMIT 15").to_pandas()
df_asp  = session.sql("""SELECT ROUND(AVG(seat_comfort),2) AS seat_comfort, ROUND(AVG(cabin_staff_service),2) AS cabin_staff, ROUND(AVG(food_and_beverages),2) AS food_bev, ROUND(AVG(ground_service),2) AS ground_service, ROUND(AVG(inflight_entertainment),2) AS ife, ROUND(AVG(wifi_and_connectivity),2) AS wifi, ROUND(AVG(value_for_money),2) AS value FROM airline_reviews_db.harmonized.airline_reviews_v""").to_pandas().T.reset_index()
df_asp.columns = ["aspect","avg_score"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(df_dist["OVERALL_RATING"], df_dist["CNT"], color=PALETTE[2], edgecolor="white")
axes[0].set_title("Rating distribution", fontsize=13, fontweight="bold"); axes[0].set_xlabel("Rating"); axes[0].set_ylabel("Reviews")

axes[1].barh(df_top["AIRLINE_NAME"][::-1], df_top["REVIEWS"][::-1], color=PALETTE[0])
axes[1].set_title("Top 15 airlines by volume", fontsize=13, fontweight="bold"); axes[1].set_xlabel("Reviews")

bars = axes[2].barh(df_asp["aspect"], df_asp["avg_score"], color=[PALETTE[2] if v>=3 else PALETTE[1] for v in df_asp["avg_score"]])
axes[2].set_xlim(0, 5.5); axes[2].set_title("Avg aspect scores", fontsize=13, fontweight="bold"); axes[2].set_xlabel("Score (1–5)")
for bar, val in zip(bars, df_asp["avg_score"]):
    axes[2].text(val+0.05, bar.get_y()+bar.get_height()/2, f"{val:.2f}", va="center", fontsize=9)

plt.suptitle("Dataset Overview", fontsize=15, fontweight="bold"); plt.tight_layout(); plt.show()

---
## Phase 3 — Translation
Apply `CORTEX.TRANSLATE` to non-English reviews (auto language detection). Only rows with non-ASCII characters are sent to the API to minimise cost.

In [ ]:
session.sql("USE SCHEMA airline_reviews_db.cortex_output").collect()
session.sql("""
    CREATE OR REPLACE TABLE airline_reviews_db.cortex_output.reviews_translated AS
    SELECT *, review AS review_original,
        CASE
            WHEN review RLIKE '.*[^\\x00-\\x7F].*'
            THEN SNOWFLAKE.CORTEX.TRANSLATE(review, '', 'en')
            ELSE review
        END AS review_english
    FROM airline_reviews_db.harmonized.airline_reviews_v
""").collect()

df_t = session.sql("SELECT review_original, review_english FROM airline_reviews_db.cortex_output.reviews_translated WHERE review_original <> review_english LIMIT 5").to_pandas()
print(f"Translation complete — {len(df_t)} sample non-English reviews shown")
df_t

---
## Phase 4a — Sentiment Scoring
`CORTEX.SENTIMENT` scores each review from **–1** (most negative) to **+1** (most positive).

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE airline_reviews_db.cortex_output.reviews_sentiment AS
    SELECT *,
        SNOWFLAKE.CORTEX.SENTIMENT(review_english) AS sentiment_score,
        CASE
            WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_english) >=  0.2 THEN 'positive'
            WHEN SNOWFLAKE.CORTEX.SENTIMENT(review_english) <= -0.2 THEN 'negative'
            ELSE 'neutral'
        END AS sentiment_label
    FROM airline_reviews_db.cortex_output.reviews_translated
""").collect()

df_sent = session.sql("""
    SELECT airline_name, COUNT(*) AS review_count,
           ROUND(AVG(sentiment_score),3) AS avg_sentiment,
           ROUND(AVG(overall_rating),2) AS avg_rating,
           SUM(CASE WHEN sentiment_label='positive' THEN 1 ELSE 0 END) AS pos_ct,
           SUM(CASE WHEN sentiment_label='negative' THEN 1 ELSE 0 END) AS neg_ct
    FROM airline_reviews_db.cortex_output.reviews_sentiment
    GROUP BY airline_name HAVING review_count >= 20
    ORDER BY avg_sentiment DESC
""").to_pandas()

top10    = df_sent.head(10)
bottom10 = df_sent.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, df_sl, title, col in [
    (axes[0], top10,    "Top 10 — avg sentiment",    PALETTE[0]),
    (axes[1], bottom10, "Bottom 10 — avg sentiment", PALETTE[1]),
]:
    ax.barh(df_sl["AIRLINE_NAME"], df_sl["AVG_SENTIMENT"], color=col)
    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title(title, fontsize=12, fontweight="bold"); ax.set_xlabel("Avg sentiment")
plt.tight_layout(); plt.show()

labels_df = session.sql("SELECT sentiment_label, COUNT(*) AS cnt FROM airline_reviews_db.cortex_output.reviews_sentiment GROUP BY sentiment_label").to_pandas()
print("Sentiment complete"); print(labels_df.to_string(index=False))

---
## Phase 4b — Classification
- **Recommend intent** (`snowflake-arctic`, zero-shot) → Likely / Unlikely / Unsure
- **Text rating** (`mistral-large`, one-shot) → awful / poor / okay / good / excellent

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE airline_reviews_db.cortex_output.reviews_rated AS
    WITH rec_cte AS (
        SELECT *,
            SNOWFLAKE.CORTEX.COMPLETE('snowflake-arctic',
                CONCAT('[INST]### Based on this airline review, will the passenger recommend ',
                    'this airline? Reply with exactly one word: Likely, Unlikely, or Unsure. ',
                    'No extra text. Review: ', review_english, ' ###[/INST]')) AS rec_raw
        FROM airline_reviews_db.cortex_output.reviews_sentiment
    ),
    rating_cte AS (
        SELECT *,
            SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
                CONCAT('[INST]### Rate this airline review with one word only: ',
                    'awful, poor, okay, good, or excellent. No extra text. ',
                    'Example: Fantastic crew, smooth flight. Rating: excellent ',
                    'Rate this review: ', review_english, ' ###[/INST]')) AS rating_raw
        FROM rec_cte
    )
    SELECT *,
        CASE WHEN UPPER(TRIM(rec_raw)) LIKE '%UNLIKELY%' THEN 'Unlikely'
             WHEN UPPER(TRIM(rec_raw)) LIKE '%LIKELY%'   THEN 'Likely'
             ELSE 'Unsure' END AS recommend_llm,
        CASE WHEN LOWER(rating_raw) LIKE '%awful%'     THEN 'awful'
             WHEN LOWER(rating_raw) LIKE '%poor%'      THEN 'poor'
             WHEN LOWER(rating_raw) LIKE '%okay%'      THEN 'okay'
             WHEN LOWER(rating_raw) LIKE '%good%'      THEN 'good'
             WHEN LOWER(rating_raw) LIKE '%excellent%' THEN 'excellent'
             ELSE 'unsure' END AS rating_llm
    FROM rating_cte
""").collect()

df_xval = session.sql("SELECT rating_llm, ROUND(AVG(overall_rating),2) AS avg_numeric, COUNT(*) AS cnt FROM airline_reviews_db.cortex_output.reviews_rated GROUP BY rating_llm ORDER BY avg_numeric DESC").to_pandas()
order = ["excellent","good","okay","poor","awful","unsure"]
df_xval["RATING_LLM"] = pd.Categorical(df_xval["RATING_LLM"], categories=order, ordered=True)
df_xval = df_xval.sort_values("RATING_LLM")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = [PALETTE[0], PALETTE[2], PALETTE[3], PALETTE[1], "#D85A30", "gray"][:len(df_xval)]
axes[0].bar(df_xval["RATING_LLM"], df_xval["AVG_NUMERIC"], color=colors)
axes[0].set_title("LLM text rating vs numeric score (cross-validation)", fontsize=12, fontweight="bold"); axes[0].set_ylabel("Avg numeric rating")
axes[1].bar(df_xval["RATING_LLM"], df_xval["CNT"], color=colors)
axes[1].set_title("LLM text rating distribution", fontsize=12, fontweight="bold"); axes[1].set_ylabel("Reviews")
plt.tight_layout(); plt.show()
print("Classification complete"); print(df_xval.to_string(index=False))

---
## Phase 5a — Aspect-Based Sentiment
Prompt `mistral-large` to extract a JSON array of aspect sentiments per review, then flatten with `LATERAL FLATTEN`.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE airline_reviews_db.cortex_output.reviews_aspect_sentiment AS
    SELECT review_id, airline_name, seat_type, traveller_type,
           overall_rating, sentiment_score, review_english,
        SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
            CONCAT('[INST]### Analyse this airline review. Identify what it says about: ',
                'seat comfort, cabin crew, food and beverage, ground service, ',
                'inflight entertainment, wifi, value for money, baggage handling, delays. ',
                'Return ONLY a valid JSON array, no markdown, no extra text. ',
                'Example: [{"category":"cabin crew","sentiment":"positive","detail":"attentive"},',
                '{"category":"wifi","sentiment":"negative","detail":"broken"}] ',
                'Review: ', review_english, ' ###[/INST]')
        ) AS aspect_json
    FROM airline_reviews_db.cortex_output.reviews_rated
""").collect()

session.sql("""
    CREATE OR REPLACE VIEW airline_reviews_db.analytics.aspect_flat_v AS
    SELECT r.review_id, r.airline_name, r.seat_type, r.traveller_type,
           r.overall_rating, r.sentiment_score,
           f.value:category::VARCHAR  AS aspect_category,
           f.value:sentiment::VARCHAR AS aspect_sentiment,
           f.value:detail::VARCHAR    AS aspect_detail
    FROM airline_reviews_db.cortex_output.reviews_aspect_sentiment r,
         LATERAL FLATTEN(INPUT => TRY_PARSE_JSON(r.aspect_json), OUTER => TRUE) f
    WHERE f.value:category IS NOT NULL
""").collect()

df_agg = session.sql("""
    SELECT airline_name, aspect_category, COUNT(*) AS mentions,
           SUM(CASE WHEN aspect_sentiment='positive' THEN 1 ELSE 0 END) AS pos,
           SUM(CASE WHEN aspect_sentiment='negative' THEN 1 ELSE 0 END) AS neg,
           ROUND(SUM(CASE WHEN aspect_sentiment='positive' THEN 1.0 WHEN aspect_sentiment='negative' THEN -1.0 ELSE 0 END)/COUNT(*),2) AS net_score
    FROM airline_reviews_db.analytics.aspect_flat_v
    WHERE aspect_category IS NOT NULL GROUP BY airline_name, aspect_category HAVING mentions >= 5
""").to_pandas()

top_al = df_agg.groupby("AIRLINE_NAME")["MENTIONS"].sum().nlargest(12).index.tolist()
pivot  = df_agg[df_agg["AIRLINE_NAME"].isin(top_al)].pivot_table(index="AIRLINE_NAME", columns="ASPECT_CATEGORY", values="NET_SCORE", aggfunc="mean")

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            linewidths=0.4, linecolor="white", cbar_kws={"label": "Net sentiment"}, ax=ax)
ax.set_title("Aspect-based sentiment heatmap — top 12 airlines", fontsize=13, fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("")
plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()
print("Aspect sentiment complete")

---
## Phase 5b — Issue Identification
Aggregate the 100 most negative reviews per airline and ask `mistral-large2` for a structured 3-bullet issue summary with recommendations.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE airline_reviews_db.cortex_output.airline_issue_summary AS
    WITH qualified AS (
        SELECT airline_name FROM airline_reviews_db.cortex_output.reviews_rated
        GROUP BY airline_name HAVING COUNT(*) >= 20
    ),
    ranked AS (
        SELECT r.*,
               ROW_NUMBER() OVER (PARTITION BY r.airline_name ORDER BY r.sentiment_score ASC) AS rn
        FROM airline_reviews_db.cortex_output.reviews_rated r
        JOIN qualified q ON r.airline_name = q.airline_name
    ),
    agg AS (
        SELECT airline_name,
               LISTAGG(review_english,' | ') WITHIN GROUP (ORDER BY sentiment_score) AS agg_text,
               ROUND(AVG(sentiment_score),3) AS avg_sentiment,
               ROUND(AVG(overall_rating),2)  AS avg_rating,
               COUNT(*) AS review_count
        FROM ranked WHERE rn <= 100 GROUP BY airline_name
    )
    SELECT airline_name, avg_sentiment, avg_rating, review_count,
        SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
            CONCAT('[INST]### CX analyst task for airline: ', airline_name, '. ',
                'From these reviews identify 3 main passenger issues. ',
                'Format: 3 bullets each with bold heading (3-5 words), one sentence issue, one sentence fix. Under 200 words. ',
                'Reviews: ', LEFT(agg_text, 8000), ' ###[/INST]')
        ) AS issue_summary
    FROM agg
""").collect()

df_iss = session.sql("SELECT airline_name, avg_rating, avg_sentiment, issue_summary FROM airline_reviews_db.cortex_output.airline_issue_summary ORDER BY avg_sentiment ASC LIMIT 5").to_pandas()
print("Issue identification complete")
for _, row in df_iss.iterrows():
    print(f"\n{'='*60}")
    print(f"  {row['AIRLINE_NAME']}  |  Rating: {row['AVG_RATING']}  |  Sentiment: {row['AVG_SENTIMENT']}")
    print(f"{'='*60}")
    print(row["ISSUE_SUMMARY"])

---
## Competitor Comparison
Side-by-side aspect sentiment for any two airlines. Change `AIRLINE_A` and `AIRLINE_B` to any airline in the dataset.

In [ ]:
AIRLINE_A = "British Airways"
AIRLINE_B = "Emirates"

df_comp = session.sql(f"""
    SELECT aspect_category,
        SUM(CASE WHEN airline_name='{AIRLINE_A}' AND aspect_sentiment='positive' THEN 1 ELSE 0 END) AS a_pos,
        SUM(CASE WHEN airline_name='{AIRLINE_A}' THEN 1 ELSE 0 END) AS a_total,
        SUM(CASE WHEN airline_name='{AIRLINE_B}' AND aspect_sentiment='positive' THEN 1 ELSE 0 END) AS b_pos,
        SUM(CASE WHEN airline_name='{AIRLINE_B}' THEN 1 ELSE 0 END) AS b_total
    FROM airline_reviews_db.analytics.aspect_flat_v
    WHERE airline_name IN ('{AIRLINE_A}','{AIRLINE_B}') AND aspect_category IS NOT NULL
    GROUP BY aspect_category
""").to_pandas()

df_comp["A_PCT"] = (df_comp["A_POS"] / df_comp["A_TOTAL"].replace(0,1) * 100).round(1)
df_comp["B_PCT"] = (df_comp["B_POS"] / df_comp["B_TOTAL"].replace(0,1) * 100).round(1)
df_comp = df_comp.sort_values("A_PCT")

x = np.arange(len(df_comp)); w = 0.35
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(x - w/2, df_comp["A_PCT"], w, label=AIRLINE_A, color=PALETTE[2])
ax.barh(x + w/2, df_comp["B_PCT"], w, label=AIRLINE_B, color=PALETTE[3])
ax.set_yticks(x); ax.set_yticklabels(df_comp["ASPECT_CATEGORY"])
ax.set_xlabel("% Positive mentions")
ax.set_title(f"Competitor comparison: {AIRLINE_A} vs {AIRLINE_B}", fontsize=13, fontweight="bold")
ax.axvline(50, color="gray", linewidth=0.8, linestyle="--", alpha=0.5); ax.legend()
plt.tight_layout(); plt.show()

---
## Aspect Radar Chart
Spider / radar chart using the 7 structured numeric aspect columns (1–5 scale).

In [ ]:
AIRLINES_RADAR = ["British Airways", "Emirates", "Singapore Airlines", "Ryanair"]
df_r = session.sql(f"""
    SELECT airline_name,
           ROUND(AVG(seat_comfort),2) AS sc, ROUND(AVG(cabin_staff_service),2) AS cs,
           ROUND(AVG(food_and_beverages),2) AS fb, ROUND(AVG(ground_service),2) AS gs,
           ROUND(AVG(inflight_entertainment),2) AS ife, ROUND(AVG(wifi_and_connectivity),2) AS wifi,
           ROUND(AVG(value_for_money),2) AS val
    FROM airline_reviews_db.harmonized.airline_reviews_v
    WHERE airline_name IN ({','.join([f"'{a}'" for a in AIRLINES_RADAR])})
    GROUP BY airline_name
""").to_pandas()

cats   = ["SC","CS","FB","GS","IFE","WIFI","VAL"]
labels = ["Seat comfort","Cabin staff","Food & bev","Ground svc","IFE","Wi-Fi","Value"]
N      = len(cats)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); angles += angles[:1]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw={"polar":True})
ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, size=10)
ax.set_ylim(0,5); ax.set_yticks([1,2,3,4,5])
ax.set_yticklabels(["1","2","3","4","5"], size=8, color="grey")
ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.6)

for i, row in df_r.iterrows():
    vals = [row[c] if pd.notna(row[c]) else 0 for c in cats] + [row[cats[0]] if pd.notna(row[cats[0]]) else 0]
    ax.plot(angles, vals, linewidth=2, color=PALETTE[i%len(PALETTE)], label=row["AIRLINE_NAME"])
    ax.fill(angles, vals, alpha=0.08, color=PALETTE[i%len(PALETTE)])

ax.legend(loc="upper right", bbox_to_anchor=(1.35,1.15), fontsize=10)
ax.set_title("Airline aspect radar (1–5 scale)", size=13, fontweight="bold", pad=20)
plt.tight_layout(); plt.show()

---
## Sentiment Trend Over Time
Monthly average sentiment for the 6 highest-volume airlines.

In [ ]:
top6 = session.sql("SELECT airline_name FROM airline_reviews_db.cortex_output.reviews_sentiment GROUP BY airline_name ORDER BY COUNT(*) DESC LIMIT 6").to_pandas()["AIRLINE_NAME"].tolist()
df_trend = session.sql(f"""
    SELECT airline_name, DATE_TRUNC('month', review_date) AS review_month,
           ROUND(AVG(sentiment_score),3) AS avg_sentiment
    FROM airline_reviews_db.cortex_output.reviews_sentiment
    WHERE airline_name IN ({','.join([f"'{a}'" for a in top6])}) AND review_date IS NOT NULL
    GROUP BY airline_name, DATE_TRUNC('month', review_date) ORDER BY airline_name, review_month
""").to_pandas()
df_trend["REVIEW_MONTH"] = pd.to_datetime(df_trend["REVIEW_MONTH"])

fig, ax = plt.subplots(figsize=(14,6))
for i, al in enumerate(top6):
    df_a = df_trend[df_trend["AIRLINE_NAME"]==al]
    if len(df_a) >= 3:
        ax.plot(df_a["REVIEW_MONTH"], df_a["AVG_SENTIMENT"], label=al, color=PALETTE[i%len(PALETTE)], linewidth=2, marker="o", markersize=3)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_title("Sentiment trend over time — top 6 airlines", fontsize=13, fontweight="bold")
ax.set_xlabel("Review date"); ax.set_ylabel("Avg sentiment"); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

---
## Phase 6 — Analytics Views for Streamlit
Create all pre-aggregated views that power the Streamlit dashboard tabs.

In [ ]:
session.sql("USE SCHEMA airline_reviews_db.analytics").collect()

view_defs = {
"v_airline_kpis": """
    SELECT airline_name, COUNT(*) AS total_reviews,
           ROUND(AVG(overall_rating),2) AS avg_rating,
           ROUND(AVG(sentiment_score),3) AS avg_sentiment,
           ROUND(SUM(CASE WHEN recommended='yes' THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS recommended_pct,
           ROUND(AVG(seat_comfort),2) AS avg_seat_comfort, ROUND(AVG(cabin_staff_service),2) AS avg_cabin_staff,
           ROUND(AVG(food_and_beverages),2) AS avg_food, ROUND(AVG(ground_service),2) AS avg_ground,
           ROUND(AVG(inflight_entertainment),2) AS avg_ife, ROUND(AVG(wifi_and_connectivity),2) AS avg_wifi,
           ROUND(AVG(value_for_money),2) AS avg_value
    FROM airline_reviews_db.cortex_output.reviews_rated GROUP BY airline_name""",
"v_sentiment_over_time": """
    SELECT airline_name, DATE_TRUNC('month', review_date) AS review_month,
           COUNT(*) AS review_count, ROUND(AVG(sentiment_score),3) AS avg_sentiment,
           ROUND(AVG(overall_rating),2) AS avg_rating
    FROM airline_reviews_db.cortex_output.reviews_rated WHERE review_date IS NOT NULL
    GROUP BY airline_name, DATE_TRUNC('month', review_date)""",
"v_aspects_by_class": """
    SELECT airline_name, seat_type,
           ROUND(AVG(seat_comfort),2) AS avg_seat_comfort, ROUND(AVG(cabin_staff_service),2) AS avg_cabin_staff,
           ROUND(AVG(food_and_beverages),2) AS avg_food, ROUND(AVG(ground_service),2) AS avg_ground,
           ROUND(AVG(inflight_entertainment),2) AS avg_ife, ROUND(AVG(wifi_and_connectivity),2) AS avg_wifi,
           ROUND(AVG(value_for_money),2) AS avg_value, COUNT(*) AS review_count
    FROM airline_reviews_db.cortex_output.reviews_rated WHERE seat_type IS NOT NULL
    GROUP BY airline_name, seat_type""",
"v_airline_issues": """
    SELECT airline_name, avg_rating, avg_sentiment, review_count, issue_summary
    FROM airline_reviews_db.cortex_output.airline_issue_summary ORDER BY avg_sentiment ASC"""
}

for vname, vbody in view_defs.items():
    session.sql(f"CREATE OR REPLACE VIEW airline_reviews_db.analytics.{vname} AS {vbody}").collect()
    print(f"  Created view: {vname}")

print("\nAll analytics views ready for Streamlit dashboard")

---
## Pipeline Summary — Row Counts

In [ ]:
layers = [
    ("raw",              "airline_reviews_db.raw.airline_reviews_raw"),
    ("harmonized",       "airline_reviews_db.harmonized.airline_reviews_v"),
    ("translated",       "airline_reviews_db.cortex_output.reviews_translated"),
    ("sentiment",        "airline_reviews_db.cortex_output.reviews_sentiment"),
    ("rated",            "airline_reviews_db.cortex_output.reviews_rated"),
    ("aspect_sentiment", "airline_reviews_db.cortex_output.reviews_aspect_sentiment"),
    ("issue_summaries",  "airline_reviews_db.cortex_output.airline_issue_summary"),
]

rows = []
for layer, tbl in layers:
    try:
        n = session.sql(f"SELECT COUNT(*) AS n FROM {tbl}").collect()[0]["N"]
        rows.append({"Layer": layer, "Table / View": tbl, "Rows": f"{n:,}"})
    except Exception as e:
        rows.append({"Layer": layer, "Table / View": tbl, "Rows": f"Error: {e}"})

print("\n" + "="*70 + "\n  PIPELINE SUMMARY\n" + "="*70)
print(pd.DataFrame(rows).to_string(index=False))
print("="*70)
print("\nNotebook complete — next step: run airline_streamlit_app.py")